# Phase 4 · Submission S5 — Knowledge Distillation: S2 (3-layer LSTM) teacher → 2-layer LSTM student

**Task:** Court Judgment Prediction (CJPE), binary outcome 0/1. **Primary metric:** Macro-F1.

**Purpose:** the knowledge-distillation (KD) experiment required in every phase — a strong *teacher* teaches a smaller, faster *student*.

**Teacher = P4_S2** (best Phase 4 model so far, Codabench 0.5615): Phase-1 Architecture C (3-layer stacked LSTM) with its top LSTM layer and final LayerNorm fine-tuned, plus the hierarchical case classifier. It is rebuilt exactly from S2's `checkpoints/best.pt` and checked on dev-clean before use.

**Student:** Architecture C cut to its **first 2 of 3 LSTM layers** (Phase-1 weights; the student's top layer = Phase-1 layer 1 is fine-tuned together with the final LayerNorm — the same layer-wise recipe as S2), with a freshly initialised case classifier of the same design. The recurrent part shrinks by one third, so encoding is about 1.5× faster.

**Distillation (Hinton et al., 2015).** The teacher is run once over every training case with its best input setting from S2 (the last 4 chunks — more context than the student sees) and its two class scores are turned into soft labels with temperature T = 2. The student, trained on the last 2 chunks, minimises

`loss = class-weighted cross-entropy (true labels) + KD_WEIGHT · T² · KL(teacher soft labels ‖ student soft labels)`

**Everything else is identical to S2** (lower-cased text, de-duplicated train, last 2 × 510-token chunks, standardised token states, token + chunk attention, 3 epochs), so the difference between S2 and S5 is the student size and the KD loss.

**Leak-free selection:** all 1,517 test cases are also in `dev.json`; they are never used. Epoch, threshold and section setting are chosen on the 1,988 **dev-clean** cases. Test labels are never read. Train only — no dev data in training.

**Reported for the analysis:** teacher vs student dev-clean Macro-F1, total / recurrent parameter counts, time and peak GPU memory to predict the test set, training time.

**Before running (Kaggle)**
- Accelerator: **GPU T4 ×2** · Add-ons → Secrets: **`HF`** · Internet on.
- Add Input → datasets: **`rohitbatra2004/phase133`** (Architecture C), **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer) and **`rohitji1/p4-s2-output-dataset`** (the S2 teacher: `checkpoints/best.pt`; its `tokens_cache.pt` is reused, so no re-tokenisation).
- Expected time ≈ 1.5 hours (teacher soft labels ≈ 15–20 min, ≈ 15 min per student epoch). Checkpoints every 500 steps and every epoch; the teacher's soft labels are cached too. To resume, add this notebook's previous output as an input and run again.

Outputs: `/kaggle/working/P4_S5_KD_S2Teacher_C2LayerStudent/` — upload `submission.zip`.

In [ ]:
# Cell 1 — Imports, random seed and device.
# Every later cell uses the names defined here.
import os, json, glob, math, time, random, shutil, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import hashlib
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE, "| GPUs visible:", torch.cuda.device_count())

In [ ]:
# Cell 2 — Paths and hyperparameters for this submission.
# Every setting lives here; no other cell hard-codes a value.
RUN_NAME = "P4_S5_KD_S2Teacher_C2LayerStudent"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
CKPT_DIR = WORK_DIR / "checkpoints"
OUT_DIR = WORK_DIR / "outputs"
for folder in (WORK_DIR, DATA_DIR, CKPT_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"

# Long-document input — identical to the S2 teacher (needed to reuse its tokens and rebuild it).
CHUNK_TOKENS = 510
TRAIN_CHUNKS = 2                 # the student trains on the last 2 chunks of every case
MAX_TAIL_CHUNKS = 8
HEAD_CHUNKS = 2
TAIL_CHARS = 60000
HEAD_CHARS = 12000

# Case-level head — identical to S2 (the teacher's head must load with these sizes).
CHUNK_DIM = 512
DROPOUT = 0.2

# Student: first 2 of the 3 Phase-1 LSTM layers; its top layer (+ final LayerNorm) is fine-tuned, as in S2.
STUDENT_LSTM_LAYERS = 2
TUNED_LSTM_LAYERS = 1

# Knowledge distillation (same temperature and weight as the Phase 3 KD submission).
KD_TEMPERATURE = 2.0
KD_WEIGHT = 1.0
TEACHER_BATCH_DOCS = 32          # cases per forward pass when the teacher labels the training set (both GPUs)

# Optimisation — same as S2.
EPOCHS = 3
BATCH_DOCS = 8
GRAD_ACCUM = 2
LR_ADAPTER = 1e-4
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
GRAD_CLIP = 1.0
SAVE_EVERY_STEPS = 500
CHECK_EVERY_STEPS = 1000
CHECK_CASES = 600
EVAL_BATCH_DOCS = 16
USE_AMP = DEVICE.type == "cuda"
print("Run:", RUN_NAME, "| outputs:", WORK_DIR)

In [ ]:
# Cell 3 — Download Phase_4_CJPE from Hugging Face using the Kaggle secret "HF".
from kaggle_secrets import UserSecretsClient
snapshot_download(repo_id=HF_REPO_ID, repo_type="dataset", token=UserSecretsClient().get_secret("HF"),
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_4_CJPE/**"])
P4_DIR = DATA_DIR / "Phase_4_CJPE"
for name in ("Train/train.json", "Train/dev.json", "Test/test.json"):
    assert (P4_DIR / name).is_file(), f"Missing {name}"
print("Phase 4 data ready in", P4_DIR)

In [ ]:
# Cell 4 — Load the splits, lower-case and normalise the text, remove duplicate training cases,
# and build "dev-clean" = dev cases whose ids are NOT in the test set. Test labels are never read.
def normalise(text):
    return " ".join(text.lower().split())

def text_key(text):
    return hashlib.md5(text.encode()).hexdigest()

raw_train = json.load(open(P4_DIR / "Train/train.json"))
raw_dev = json.load(open(P4_DIR / "Train/dev.json"))
raw_test = json.load(open(P4_DIR / "Test/test.json"))

train_cases, seen = [], set()
for d in raw_train:
    text = normalise(d["text"])
    key = text_key(text)
    if key in seen:
        continue
    seen.add(key)
    train_cases.append({"id": str(d["id"]), "text": text, "label": int(d["label"])})

test_cases = [{"id": str(d["id"]), "text": normalise(d["text"])} for d in raw_test]       # labels deliberately ignored
test_ids = {c["id"] for c in test_cases}
dev_all = [{"id": str(d["id"]), "text": normalise(d["text"]), "label": int(d["label"])} for d in raw_dev]
dev_cases = [c for c in dev_all if c["id"] not in test_ids]
del raw_train, raw_dev, raw_test

print(f"train: {len(train_cases):,} unique cases (duplicates removed) | labels {Counter(c['label'] for c in train_cases)}")
print(f"dev  : {len(dev_all):,} cases, of which {len(dev_all) - len(dev_cases):,} are also in test and are excluded "
      f"→ dev-clean {len(dev_cases):,} | labels {Counter(c['label'] for c in dev_cases)}")
print(f"test : {len(test_cases):,} cases")

In [ ]:
# Cell 5 — Tokens of every case (last 8×510 and first 2×510 BPE tokens). Reused from the S2 output when attached
# (same text processing), after checking a few cases against a fresh encoding; otherwise tokenised here (≈ 10 min).
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
PAD_ID, BOS_ID, EOS_ID = (tokenizer.token_to_id(t) for t in ("[PAD]", "[BOS]", "[EOS]"))
assert (PAD_ID, BOS_ID, EOS_ID) == (0, 2, 3)
TAIL_LEN, HEAD_LEN = MAX_TAIL_CHUNKS * CHUNK_TOKENS, HEAD_CHUNKS * CHUNK_TOKENS
SPLITS = (("train", train_cases), ("dev", dev_cases), ("test", test_cases))

def encode_cases(cases):
    tails, heads = [], []
    for start in range(0, len(cases), 512):
        batch = cases[start:start + 512]
        tail_enc = tokenizer.encode_batch([c["text"][-TAIL_CHARS:] for c in batch])
        head_enc = tokenizer.encode_batch([c["text"][:HEAD_CHARS] for c in batch])
        for c, t, h in zip(batch, tail_enc, head_enc):
            ids = t.ids[1:] if len(c["text"]) > TAIL_CHARS else t.ids      # drop a possibly cut first token
            tails.append(np.array(ids[-TAIL_LEN:], dtype=np.uint16))
            heads.append(np.array(h.ids[:HEAD_LEN], dtype=np.uint16))
    return tails, heads

def cache_matches(tokens):
    """Same case counts and identical tokens for the first and last 3 cases of every split."""
    for name, cases in SPLITS:
        if len(tokens[name][0]) != len(cases):
            return False
        for j in (0, 1, 2, len(cases) - 3, len(cases) - 2, len(cases) - 1):
            tail, head = encode_cases([cases[j]])
            if not (np.array_equal(tail[0], tokens[name][0][j]) and np.array_equal(head[0], tokens[name][1][j])):
                return False
    return True

cache = WORK_DIR / "tokens_cache.pt"
tokens = None
candidates = ([cache] if cache.exists() else []) + [Path(p) for p in sorted(glob.glob("/kaggle/input/**/tokens_cache.pt", recursive=True))]
for path in candidates:
    loaded = torch.load(path, weights_only=False)
    if cache_matches(loaded):
        tokens = loaded
        print("Reusing tokens from", path)
        break
    print("Tokens in", path, "do not match this data — ignored")
if tokens is None:
    start_time = time.time()
    tokens = {name: encode_cases(cases) for name, cases in SPLITS}
    print(f"Tokenised all cases in {(time.time() - start_time) / 60:.1f} min")
if not cache.exists():
    torch.save(tokens, cache)
for name, cases in SPLITS:
    lengths = np.array([len(t) for t in tokens[name][0]])
    print(f"{name}: {len(cases):,} cases | tail tokens kept median {int(np.median(lengths))}")

In [ ]:
# Cell 6 — Phase-1 Architecture C (stacked LSTM): rebuild it, load its weights strictly, and a function that
# returns a fresh copy with only the first N LSTM layers (N = 3 for the teacher, N = 2 for the student).
c_paths = sorted(glob.glob("/kaggle/input/**/architecture_C/model_C_state_dict.pt", recursive=True))
assert c_paths, "Add the dataset rohitbatra2004/phase133 as an input."
C_ROOT = Path(c_paths[0]).parent
C_METADATA = json.load(open(C_ROOT / "model_C_metadata.json"))
C_CONFIG = C_METADATA["model_config"]
HIDDEN = C_CONFIG["hidden_size"]

class ArchitectureCBackbone(nn.Module):
    """Phase-1 stacked LSTM language model without the vocabulary projection.
    Returns the normalised contextual state of every token: [batch, tokens, hidden_size]."""
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["embedding_dim"])
        self.lstm = nn.LSTM(cfg["embedding_dim"], cfg["hidden_size"], cfg["num_layers"],
                            dropout=cfg["dropout"], bias=cfg["use_bias"], batch_first=True)
        self.final_norm = nn.LayerNorm(cfg["hidden_size"])
        # Kept only so that the saved Phase-1 state dict loads strictly; unused here.
        self.output_bias = nn.Parameter(torch.zeros(cfg["vocab_size"]))

    def forward(self, input_ids):
        states, _ = self.lstm(self.token_embedding(input_ids))
        return self.final_norm(states)

saved = torch.load(c_paths[0], map_location="cpu", weights_only=True)
phase1_state = saved["state_dict"] if "state_dict" in saved else saved

def lstm_layer_of(name):
    """'lstm.weight_ih_l1' → 1; None for parameters outside the LSTM."""
    return int(name.rsplit("_l", 1)[1]) if name.startswith("lstm.") else None

def make_backbone(num_layers):
    """Fresh Phase-1 backbone with its first `num_layers` LSTM layers, every weight loaded strictly, all frozen."""
    bb = ArchitectureCBackbone(dict(C_CONFIG, num_layers=num_layers))
    kept = {k: v for k, v in phase1_state.items() if lstm_layer_of(k) is None or lstm_layer_of(k) < num_layers}
    bb.load_state_dict(kept, strict=True)
    for p in bb.parameters():
        p.requires_grad_(False)
    return bb

n_teacher_backbone = sum(p.numel() for p in make_backbone(C_CONFIG["num_layers"]).parameters())
assert n_teacher_backbone == int(C_METADATA["parameter_count"]), "Parameter count differs from Phase-1 metadata"
print(json.dumps(C_CONFIG, indent=2))
print(f"Loaded Phase-1 Architecture C from {C_ROOT} — {n_teacher_backbone:,} parameters")

In [ ]:
# Cell 7 — Case classifier (same design as S2, used by both teacher and student), batching and evaluation helpers.
# Standardised token states → token attention in each chunk → attention over chunks → 2 classes.
class CaseClassifier(nn.Module):
    def __init__(self, backbone, hidden):
        super().__init__()
        self.backbone = backbone
        self.token_score = nn.Sequential(nn.Linear(hidden, 256), nn.Tanh(), nn.Linear(256, 1))   # token-level attention
        self.chunk_proj = nn.Sequential(nn.Linear(3 * hidden, CHUNK_DIM), nn.GELU(), nn.Dropout(DROPOUT))
        self.register_buffer("state_mean", torch.zeros(hidden))     # per-dimension standardisation of token states
        self.register_buffer("state_std", torch.ones(hidden))
        self.chunk_position = nn.Embedding(MAX_TAIL_CHUNKS, CHUNK_DIM)      # 0 = last chunk of the case, 1 = one before …
        nn.init.normal_(self.chunk_position.weight, std=0.02)
        self.attn_score = nn.Sequential(nn.Linear(CHUNK_DIM, CHUNK_DIM), nn.Tanh(), nn.Linear(CHUNK_DIM, 1))
        self.case_norm = nn.LayerNorm(CHUNK_DIM)
        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(CHUNK_DIM, 2)

    def forward(self, ids, chunk_mask, chunk_pos):
        """ids [B, K, T], chunk_mask [B, K] (True = real chunk), chunk_pos [B, K] → (logits [B, 2], attention [B, K])."""
        B, K, T = ids.shape
        flat_ids, valid = ids.view(B * K, T), chunk_mask.view(-1)
        token_mask = flat_ids[valid].ne(PAD_ID)
        states = ((self.backbone(flat_ids[valid]).float() - self.state_mean) / self.state_std).to(self.state_mean.dtype)
        m = token_mask.unsqueeze(-1).to(states.dtype)
        mean = (states * m).sum(1) / m.sum(1).clamp(min=1.0)
        last = states[torch.arange(states.size(0), device=states.device), token_mask.sum(1) - 1]
        token_weights = torch.softmax(self.token_score(states).squeeze(-1).float().masked_fill(~token_mask, -1e4), dim=-1)
        attended = (token_weights.unsqueeze(-1) * states.float()).sum(1).to(states.dtype)
        vecs = self.chunk_proj(torch.cat([mean, last, attended], dim=-1))
        chunks = torch.zeros(B * K, CHUNK_DIM, device=ids.device, dtype=vecs.dtype)
        chunks[valid] = vecs
        chunks = chunks.view(B, K, CHUNK_DIM) + self.chunk_position(chunk_pos).to(vecs.dtype)
        scores = self.attn_score(chunks).squeeze(-1).float().masked_fill(~chunk_mask, -1e4)
        weights = torch.softmax(scores, dim=-1)
        case_vec = (weights.unsqueeze(-1) * chunks.float()).sum(1)
        return self.classifier(self.dropout(self.case_norm(case_vec))), weights

def case_chunks(index, split, part, n_chunks):
    tail, head = tokens[split][0][index], tokens[split][1][index]
    if part == "tail":
        ids = tail[-n_chunks * CHUNK_TOKENS:]
        pieces = [ids[max(0, end - CHUNK_TOKENS):end] for end in range(len(ids), 0, -CHUNK_TOKENS)][::-1]
    else:
        ids = head[:n_chunks * CHUNK_TOKENS]
        pieces = [ids[s:s + CHUNK_TOKENS] for s in range(0, len(ids), CHUNK_TOKENS)]
    pieces = [p for p in pieces if len(p) > 0] or [np.array([], dtype=np.uint16)]
    return [[BOS_ID] + p.tolist() + [EOS_ID] for p in pieces]

def make_batch(indices, split, part, n_chunks):
    per_case = [case_chunks(i, split, part, n_chunks) for i in indices]
    K = max(len(c) for c in per_case)
    T = max(len(x) for c in per_case for x in c)
    ids = torch.full((len(indices), K, T), PAD_ID, dtype=torch.long)
    mask = torch.zeros(len(indices), K, dtype=torch.bool)
    pos = torch.zeros(len(indices), K, dtype=torch.long)
    for b, chunks in enumerate(per_case):
        for k, x in enumerate(chunks):
            ids[b, k, :len(x)] = torch.tensor(x)
            mask[b, k] = True
            pos[b, k] = (len(chunks) - 1 - k) if part == "tail" else min(k, MAX_TAIL_CHUNKS - 1)
    return ids.to(DEVICE), mask.to(DEVICE), pos.to(DEVICE)

@torch.no_grad()
def predict_logits(net, split, part="tail", n_chunks=TRAIN_CHUNKS, limit=None, batch_docs=EVAL_BATCH_DOCS):
    """Class scores [cases, 2] for a split; `net` may be a model or its DataParallel wrapper."""
    net.eval()
    n = len(tokens[split][0]) if limit is None else min(limit, len(tokens[split][0]))
    out = []
    for start in range(0, n, batch_docs):
        batch = make_batch(list(range(start, min(start + batch_docs, n))), split, part, n_chunks)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits, _ = net(*batch)
        out.append(logits.float().cpu())
    return torch.cat(out)

def predict_probs(net, split, part="tail", n_chunks=TRAIN_CHUNKS, limit=None):
    return torch.softmax(predict_logits(net, split, part, n_chunks, limit), -1)[:, 1].numpy()

def scores_at(gold, probs, threshold):
    pred = (probs >= threshold).astype(int)
    return {"macro_f1": f1_score(gold, pred, average="macro", zero_division=0),
            "macro_p": precision_score(gold, pred, average="macro", zero_division=0),
            "macro_r": recall_score(gold, pred, average="macro", zero_division=0), "threshold": float(threshold)}

def evaluate(net, part="tail", n_chunks=TRAIN_CHUNKS, limit=None):
    probs = predict_probs(net, "dev", part, n_chunks, limit)
    gold = np.array([c["label"] for c in dev_cases])[:len(probs)]
    at_half = scores_at(gold, probs, 0.5)
    best = max((scores_at(gold, probs, t) for t in np.arange(0.20, 0.81, 0.01)), key=lambda s: s["macro_f1"])
    return {"at_0.5": at_half, "tuned": best}, probs

def timed_test_prediction(net, part, n_chunks):
    """Test-set probabilities, wall-clock seconds and peak GPU memory (GB) of one full prediction pass."""
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    start = time.time()
    probs = predict_probs(net, "test", part, n_chunks)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0
    return probs, time.time() - start, peak

train_labels = torch.tensor([c["label"] for c in train_cases])
counts = torch.bincount(train_labels, minlength=2).float()
CLASS_WEIGHTS = (counts.sum() / (2 * counts)).to(DEVICE)               # balanced weights for the 59/41 train split
N_GPUS = torch.cuda.device_count()
print("Helpers ready | class weights:", CLASS_WEIGHTS.tolist(), "| GPUs:", max(N_GPUS, 1))

In [ ]:
# Cell 8 — Teacher: rebuild P4_S2 from its best.pt, check it on dev-clean, time it on test,
# and compute its soft labels for every training case (cached in teacher_train_logits.pt).
def find_teacher_checkpoint():
    top = f"backbone.lstm.weight_ih_l{C_CONFIG['num_layers'] - 1}"      # S2 fine-tuned the top LSTM layer
    for path in sorted(glob.glob("/kaggle/input/**/checkpoints/best.pt", recursive=True)):
        if RUN_NAME in path:
            continue
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        if isinstance(ckpt, dict) and top in ckpt.get("trainable", {}):
            return Path(path), ckpt
    raise AssertionError("Add the S2 output dataset (rohitji1/p4-s2-output-dataset, with checkpoints/best.pt) as an input.")

teacher_path, teacher_ckpt = find_teacher_checkpoint()
teacher = CaseClassifier(make_backbone(C_CONFIG["num_layers"]), HIDDEN).to(DEVICE)
missing, unexpected = teacher.load_state_dict(teacher_ckpt["trainable"], strict=False)
assert not unexpected, f"Unexpected teacher keys: {unexpected[:5]}"
must_have = [n for n, _ in teacher.named_parameters() if not n.startswith("backbone.")] + ["state_mean", "state_std"]
assert all(n in teacher_ckpt["trainable"] for n in must_have), "S2 checkpoint is missing head / standardisation weights"
for p in teacher.parameters():
    p.requires_grad_(False)
teacher.eval()
print(f"Teacher loaded from {teacher_path} (S2 best epoch {teacher_ckpt.get('epoch')})")

# The teacher's best input setting, as chosen on dev-clean by S2 (falls back to the last 4 chunks).
s2_summary = teacher_path.parent.parent / "outputs" / "run_summary.json"
setting = json.load(open(s2_summary))["test_setting"] if s2_summary.exists() else {"part": "tail", "chunks": 4}
TEACHER_PART, TEACHER_CHUNKS = setting["part"], int(setting["chunks"])
teacher_dev, teacher_dev_probs = evaluate(teacher, TEACHER_PART, TEACHER_CHUNKS)
print(f"Teacher on dev-clean ({TEACHER_PART} {TEACHER_CHUNKS} chunks): Macro-F1 {teacher_dev['tuned']['macro_f1']:.4f} "
      f"at threshold {teacher_dev['tuned']['threshold']:.2f} | spread {teacher_dev_probs.std():.3f}")
assert teacher_dev_probs.std() > 0.01, "The teacher gives the same answer for every case — wrong checkpoint?"
_, teacher_test_seconds, teacher_test_peak = timed_test_prediction(teacher, TEACHER_PART, TEACHER_CHUNKS)
print(f"Teacher test prediction: {teacher_test_seconds:.1f} s, peak GPU memory {teacher_test_peak:.2f} GB")

soft_cache = WORK_DIR / "teacher_train_logits.pt"
if not soft_cache.exists():
    previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/teacher_train_logits.pt", recursive=True))
    if previous:
        shutil.copy(previous[0], soft_cache)
if soft_cache.exists():
    teacher_train_logits = torch.load(soft_cache)
    print("Loaded cached teacher logits from", soft_cache)
else:
    start_time = time.time()
    runner = nn.DataParallel(teacher) if N_GPUS > 1 else teacher
    teacher_train_logits = predict_logits(runner, "train", TEACHER_PART, TEACHER_CHUNKS, batch_docs=TEACHER_BATCH_DOCS)
    torch.save(teacher_train_logits, soft_cache)
    print(f"Teacher labelled {len(teacher_train_logits):,} training cases in {(time.time() - start_time) / 60:.1f} min")
assert teacher_train_logits.shape == (len(train_cases), 2)
teacher_soft = torch.softmax(teacher_train_logits / KD_TEMPERATURE, -1)
teacher_train_pred = teacher_train_logits.argmax(-1).numpy()
print(f"Teacher on train: Macro-F1 {f1_score(train_labels.numpy(), teacher_train_pred, average='macro'):.4f} | "
      f"mean soft label for class 1 {teacher_soft[:, 1].mean():.3f} (T = {KD_TEMPERATURE})")

n_teacher_total = sum(p.numel() for p in teacher.parameters())
n_teacher_recurrent = sum(p.numel() for n, p in teacher.backbone.named_parameters() if n.startswith("lstm."))
del teacher
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# Cell 9 — Student: Phase-1 C with only its first 2 LSTM layers; its top layer and the final LayerNorm are trained
# (layer-wise fine-tuning, as in S2), plus a new case classifier. Standardisation statistics from training chunks.
student_backbone = make_backbone(STUDENT_LSTM_LAYERS)
tuned_layers = [STUDENT_LSTM_LAYERS - 1 - i for i in range(TUNED_LSTM_LAYERS)]
for name, p in student_backbone.named_parameters():
    if lstm_layer_of(name) in tuned_layers or name.startswith("final_norm"):
        p.requires_grad_(True)
for i in range(STUDENT_LSTM_LAYERS):                       # the kept layers are exact copies of Phase-1 layers 0 and 1
    assert torch.equal(student_backbone.lstm.get_parameter(f"weight_hh_l{i}").detach(), phase1_state[f"lstm.weight_hh_l{i}"])

@torch.no_grad()
def set_state_statistics(model, n_cases=512):
    """Per-dimension mean / std of the (initially Phase-1) backbone token states over training chunks."""
    model.eval()
    picked = random.Random(SEED).sample(range(len(train_cases)), min(n_cases, len(train_cases)))
    total, total_sq, count = 0.0, 0.0, 0
    for start in range(0, len(picked), 16):
        ids, mask, _ = make_batch(picked[start:start + 16], "train", "tail", 1)
        flat = ids[:, 0]
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            states = model.backbone(flat).float()[flat.ne(PAD_ID)]
        total, total_sq, count = total + states.sum(0), total_sq + (states ** 2).sum(0), count + states.size(0)
    mean = total / count
    model.state_mean.copy_(mean); model.state_std.copy_((total_sq / count - mean ** 2).clamp(min=1e-6).sqrt())

torch.manual_seed(SEED)
model = CaseClassifier(student_backbone, HIDDEN).to(DEVICE)
set_state_statistics(model)
TRAINABLE_NAMES = [n for n, p in model.named_parameters() if p.requires_grad]
SAVED_NAMES = TRAINABLE_NAMES + ["state_mean", "state_std"]
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_student_total = sum(p.numel() for p in model.parameters())
n_student_recurrent = sum(p.numel() for n, p in model.backbone.named_parameters() if n.startswith("lstm."))
train_model = nn.DataParallel(model) if N_GPUS > 1 else model
print(f"Student: LSTM layers 0–{STUDENT_LSTM_LAYERS - 1}, tuned layer(s) {tuned_layers} + final LayerNorm")
print(f"Trainable {n_trainable:,} of {n_student_total:,} parameters ({100 * n_trainable / n_student_total:.2f}%)")
print(f"Size: student {n_student_total:,} vs teacher {n_teacher_total:,} total parameters "
      f"({100 * n_student_total / n_teacher_total:.1f}%); recurrent part {n_student_recurrent:,} vs {n_teacher_recurrent:,} "
      f"({100 * n_student_recurrent / n_teacher_recurrent:.1f}%)")

In [ ]:
# Cell 10 — Optimiser, learning-rate schedule and checkpoint resume (every 500 steps and every epoch).
adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone.")]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("backbone.")]
optimizer = torch.optim.AdamW([{"params": adapter_params, "lr": LR_ADAPTER}, {"params": head_params, "lr": LR_HEAD}],
                              weight_decay=WEIGHT_DECAY)
steps_per_epoch = math.ceil(len(train_cases) / BATCH_DOCS)
total_updates = EPOCHS * math.ceil(steps_per_epoch / GRAD_ACCUM)
warmup_updates = int(WARMUP_FRACTION * total_updates)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda u: (u + 1) / max(warmup_updates, 1) if u < warmup_updates
                                              else max(0.0, (total_updates - u) / max(total_updates - warmup_updates, 1)))
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def trainable_state():
    full = model.state_dict()
    return {n: full[n].detach().cpu() for n in SAVED_NAMES}

LATEST, BEST = CKPT_DIR / "latest.pt", CKPT_DIR / "best.pt"
if not LATEST.exists():
    previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/checkpoints/latest.pt", recursive=True))
    if previous:
        shutil.copy(previous[0], LATEST)
        if Path(previous[0]).with_name("best.pt").exists():
            shutil.copy(Path(previous[0]).with_name("best.pt"), BEST)
        print("Copied checkpoints from", Path(previous[0]).parent)

state = {"epoch": 0, "step": 0, "best_f1": -1.0, "best_epoch": 0, "history": [], "train_seconds": 0.0, "train_peak_gb": 0.0}
if LATEST.exists():
    ckpt = torch.load(LATEST, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["trainable"], strict=False)
    optimizer.load_state_dict(ckpt["optimizer"]); scheduler.load_state_dict(ckpt["scheduler"]); scaler.load_state_dict(ckpt["scaler"])
    state = ckpt["state"]
    print(f"Resumed at epoch {state['epoch'] + 1}, step {state['step']} (best dev-clean Macro-F1 {state['best_f1']:.4f})")
else:
    print("No checkpoint found — the student starts from the Phase-1 weights.")
print(f"{steps_per_epoch} steps per epoch, {total_updates} optimiser updates in total")

In [ ]:
# Cell 11 — Student training with the KD loss: class-weighted CE (true labels) + KD_WEIGHT · T² · KL(teacher ‖ student).
# Every epoch ends with a dev-clean evaluation; best.pt keeps the epoch with the highest tuned-threshold Macro-F1.
def save_latest():
    torch.save({"trainable": trainable_state(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "state": state}, LATEST)

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
while state["epoch"] < EPOCHS:
    epoch = state["epoch"]
    order = list(range(len(train_cases)))
    random.Random(SEED + epoch).shuffle(order)                         # same order after a resume
    train_model.train()
    start_time, ce_sum, kd_sum, loss_count = time.time(), 0.0, 0.0, 0
    for step in range(state["step"], steps_per_epoch):
        batch_idx = order[step * BATCH_DOCS:(step + 1) * BATCH_DOCS]
        ids, mask, pos = make_batch(batch_idx, "train", "tail", TRAIN_CHUNKS)
        labels = train_labels[batch_idx].to(DEVICE)
        soft = teacher_soft[batch_idx].to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits, _ = train_model(ids, mask, pos)
        ce = F.cross_entropy(logits.float(), labels, weight=CLASS_WEIGHTS)
        kd = F.kl_div(F.log_softmax(logits.float() / KD_TEMPERATURE, -1), soft, reduction="batchmean") * KD_TEMPERATURE ** 2
        scaler.scale((ce + KD_WEIGHT * kd) / GRAD_ACCUM).backward()
        ce_sum, kd_sum, loss_count = ce_sum + ce.item(), kd_sum + kd.item(), loss_count + 1
        if (step + 1) % GRAD_ACCUM == 0 or step + 1 == steps_per_epoch:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(adapter_params + head_params, GRAD_CLIP)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        state["step"] = step + 1
        if state["step"] % SAVE_EVERY_STEPS == 0:
            state["train_seconds"] += time.time() - start_time; start_time = time.time()
            save_latest()
            print(f"  epoch {epoch + 1} step {state['step']}/{steps_per_epoch} | CE {ce_sum / loss_count:.4f} | "
                  f"KD {kd_sum / loss_count:.4f}", flush=True)
        if state["step"] % CHECK_EVERY_STEPS == 0:
            quick, quick_p = evaluate(model, limit=CHECK_CASES)
            print(f"    quick dev check ({CHECK_CASES} cases): Macro-F1 {quick['tuned']['macro_f1']:.4f} | "
                  f"spread of P(label=1) {quick_p.std():.3f}", flush=True)
            train_model.train()
    state["train_seconds"] += time.time() - start_time
    if DEVICE.type == "cuda":
        state["train_peak_gb"] = max(state["train_peak_gb"], torch.cuda.max_memory_allocated() / 1e9)
    dev, dev_p = evaluate(model)
    record = {"epoch": epoch + 1, "train_ce": ce_sum / max(loss_count, 1), "train_kd": kd_sum / max(loss_count, 1),
              "dev_prob_std": float(dev_p.std()), **{f"dev_{k}": v for k, v in dev.items()}}
    state["history"].append(record)
    mark = ""
    if dev["tuned"]["macro_f1"] > state["best_f1"]:
        state.update(best_f1=dev["tuned"]["macro_f1"], best_epoch=epoch + 1)
        torch.save({"trainable": trainable_state(), "epoch": epoch + 1, "dev": dev}, BEST)
        mark = "  <- best"
    state["epoch"], state["step"] = epoch + 1, 0
    save_latest()
    print(f"Epoch {epoch + 1} | CE {record['train_ce']:.4f} | KD {record['train_kd']:.4f} | dev-clean Macro-F1 "
          f"{dev['at_0.5']['macro_f1']:.4f} at 0.5, {dev['tuned']['macro_f1']:.4f} at threshold {dev['tuned']['threshold']:.2f} | "
          f"spread of P(label=1) {dev_p.std():.3f}{mark}")
    if dev_p.std() < 0.01:
        print("  WARNING: almost identical predictions for every case — the model has not found a signal yet.")
print(f"Best dev-clean Macro-F1 {state['best_f1']:.4f} at epoch {state['best_epoch']} | "
      f"training time {state['train_seconds'] / 60:.1f} min")

In [ ]:
# Cell 12 — Best student epoch: dev-clean report, confusion matrix, error groups, and agreement with the teacher.
best = torch.load(BEST, map_location="cpu", weights_only=False)
model.load_state_dict(best["trainable"], strict=False)
dev_result, dev_probs = evaluate(model)
THRESHOLD = dev_result["tuned"]["threshold"]
gold = np.array([c["label"] for c in dev_cases])
pred = (dev_probs >= THRESHOLD).astype(int)
teacher_pred = (teacher_dev_probs >= teacher_dev["tuned"]["threshold"]).astype(int)
print(f"Best epoch {best['epoch']} — student dev-clean Macro-F1 {dev_result['tuned']['macro_f1']:.4f} at threshold "
      f"{THRESHOLD:.2f} (P {dev_result['tuned']['macro_p']:.4f}, R {dev_result['tuned']['macro_r']:.4f}); "
      f"{dev_result['at_0.5']['macro_f1']:.4f} at 0.5")
print(f"Teacher dev-clean Macro-F1 {teacher_dev['tuned']['macro_f1']:.4f} → student keeps "
      f"{100 * dev_result['tuned']['macro_f1'] / teacher_dev['tuned']['macro_f1']:.1f}% | "
      f"student and teacher agree on {100 * np.mean(pred == teacher_pred):.1f}% of dev-clean cases")
print(classification_report(gold, pred, digits=4, zero_division=0))
cm = confusion_matrix(gold, pred)
print("Confusion matrix [rows = gold 0/1, columns = predicted 0/1]:", cm.tolist())

lengths = np.array([len(c["text"].split()) for c in dev_cases])
groups = {"true negatives": (gold == 0) & (pred == 0), "false positives": (gold == 0) & (pred == 1),
          "false negatives": (gold == 1) & (pred == 0), "true positives": (gold == 1) & (pred == 1)}
error_stats = {name: {"count": int(g.sum()), "median_words": int(np.median(lengths[g])) if g.any() else 0,
                      "mean_confidence": float(np.mean(np.abs(dev_probs[g] - 0.5) + 0.5)) if g.any() else 0.0}
               for name, g in groups.items()}
for name, s in error_stats.items():
    print(f"  {name:16s}: {s['count']:4d} cases, median length {s['median_words']:5d} words, mean confidence {s['mean_confidence']:.3f}")
json.dump({"dev": dev_result, "teacher_dev": teacher_dev, "best_epoch": best["epoch"], "history": state["history"],
           "confusion_matrix": cm.tolist(), "error_stats": error_stats,
           "teacher_student_agreement": float(np.mean(pred == teacher_pred))},
          open(OUT_DIR / "dev_metrics.json", "w"), indent=2)

In [ ]:
# Cell 13 — Section analysis on dev-clean: first 2 chunks vs last 2 / 4 / 8 chunks (same trained student).
# The best setting on dev-clean (with its own tuned threshold) is used for the test predictions.
section_results = {}
for part, n_chunks in (("head", 2), ("tail", 2), ("tail", 4), ("tail", 8)):
    result, _ = evaluate(model, part, n_chunks)
    section_results[f"{'first' if part == 'head' else 'last'}_{n_chunks}"] = {"part": part, "n_chunks": n_chunks, **result}
    print(f"  {('first' if part == 'head' else 'last')} {n_chunks} chunks: Macro-F1 {result['tuned']['macro_f1']:.4f} "
          f"(threshold {result['tuned']['threshold']:.2f}), {result['at_0.5']['macro_f1']:.4f} at 0.5")
best_section = max(section_results, key=lambda k: section_results[k]["tuned"]["macro_f1"])
TEST_PART, TEST_CHUNKS = section_results[best_section]["part"], section_results[best_section]["n_chunks"]
TEST_THRESHOLD = section_results[best_section]["tuned"]["threshold"]
print(f"Chosen for test: {best_section} (threshold {TEST_THRESHOLD:.2f})")
json.dump(section_results, open(OUT_DIR / "section_analysis.json", "w"), indent=2)

In [ ]:
# Cell 14 — Student speed vs teacher on the same input setting, then test predictions, format check and submission.zip.
# prediction.json is a JSON list of {"id": ..., "pred": 0 or 1}, one entry per test case, in test-file order.
_, student_test_seconds, student_test_peak = timed_test_prediction(model, TEACHER_PART, TEACHER_CHUNKS)
print(f"Test prediction ({TEACHER_PART} {TEACHER_CHUNKS} chunks): teacher {teacher_test_seconds:.1f} s / "
      f"{teacher_test_peak:.2f} GB, student {student_test_seconds:.1f} s / {student_test_peak:.2f} GB "
      f"→ {teacher_test_seconds / student_test_seconds:.2f}× faster")
test_probs = predict_probs(model, "test", TEST_PART, TEST_CHUNKS)
prediction = [{"id": c["id"], "pred": int(p >= TEST_THRESHOLD)} for c, p in zip(test_cases, test_probs)]
assert [p["id"] for p in prediction] == [c["id"] for c in test_cases] and len(set(test_ids)) == len(prediction)
assert all(p["pred"] in (0, 1) and isinstance(p["pred"], int) for p in prediction)
json.dump(prediction, open(WORK_DIR / "prediction.json", "w"), indent=2)
with zipfile.ZipFile(WORK_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(WORK_DIR / "prediction.json", arcname="prediction.json")
with zipfile.ZipFile(WORK_DIR / "submission.zip") as zf:
    assert zf.namelist() == ["prediction.json"]
print(f"submission.zip: {len(prediction)} cases | predicted label counts {dict(Counter(p['pred'] for p in prediction))}")

In [ ]:
# Cell 15 — Run summary for the README, the experiment log and results/submissions.csv.
summary = {
    "run": RUN_NAME, "technique": "Knowledge distillation: P4_S2 teacher (3-layer LSTM) -> 2-layer LSTM student "
                                  "(top layer + final LayerNorm tuned)",
    "kd_temperature": KD_TEMPERATURE, "kd_weight": KD_WEIGHT, "teacher_checkpoint": str(teacher_path),
    "teacher_input": {"part": TEACHER_PART, "chunks": TEACHER_CHUNKS},
    "teacher_total_params": n_teacher_total, "student_total_params": n_student_total,
    "teacher_recurrent_params": n_teacher_recurrent, "student_recurrent_params": n_student_recurrent,
    "student_trainable_params": n_trainable,
    "teacher_test_seconds": round(teacher_test_seconds, 1), "student_test_seconds": round(student_test_seconds, 1),
    "teacher_test_peak_gb": round(teacher_test_peak, 2), "student_test_peak_gb": round(student_test_peak, 2),
    "train_cases": len(train_cases), "dev_clean_cases": len(dev_cases), "epochs": state["epoch"],
    "best_epoch": state["best_epoch"], "train_minutes": round(state["train_seconds"] / 60, 1),
    "train_peak_gpu_mem_gb": round(state["train_peak_gb"], 2),
    "teacher_dev_clean": teacher_dev, "student_dev_clean": dev_result,
    "sections": {k: v["tuned"]["macro_f1"] for k, v in section_results.items()},
    "test_setting": {"part": TEST_PART, "chunks": TEST_CHUNKS, "threshold": TEST_THRESHOLD},
}
json.dump(summary, open(OUT_DIR / "run_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))